# Advanced Python Metaclasses — Problems with Complete Solutions

This notebook is a problem-driven deep dive into Python metaclasses.

It focuses on:

- how Python creates classes,
- `type`, `__prepare__`, `__new__`, `__init__`, and `__call__`,
- class namespace transformation,
- class-level validation,
- registries,
- descriptors and `__set_name__`,
- metaclass keyword arguments,
- abstract base classes and metaclass conflicts,
- method-signature enforcement,
- instance caching,
- inheritance-aware field collection,
- alternatives such as class decorators and `__init_subclass__`,
- and a capstone mini schema/ORM-style system.

Every exercise includes a complete solution and executable assertions.

> Best practice: metaclasses are powerful, but they are usually the *last* tool to reach for. Prefer normal inheritance, composition, decorators, descriptors, or `__init_subclass__` when they solve the problem cleanly.


## 0. Mental model: what a metaclass actually does

For a normal class statement:

```python
class Example(Base):
    x = 10
```

Python roughly performs these conceptual steps:

1. determine the metaclass,
2. call `metaclass.__prepare__(name, bases, **kwargs)` to get the class namespace,
3. execute the class body inside that namespace,
4. call the metaclass to create the class,
5. bind the resulting class object to `Example`.

Because metaclasses are classes, the call itself follows the usual instance-creation protocol:

```text
Metaclass.__call__
    -> Metaclass.__new__
    -> Metaclass.__init__
```

The "instance" created by a metaclass is itself a class.


In [1]:
# Basic sanity check: classes are objects, and normal classes are instances of `type`.

class BasicExample:
    pass

assert isinstance(BasicExample, type)
assert type(BasicExample) is type

obj = BasicExample()
assert type(obj) is BasicExample

print("type(BasicExample):", type(BasicExample))
print("type(obj):", type(obj))


type(BasicExample): <class 'type'>
type(obj): <class '__main__.BasicExample'>


# Problem 1 — Trace the entire class-creation and instance-creation lifecycle

Create a metaclass named `TraceMeta` that records the order in which these hooks run:

- `__prepare__`
- `__new__`
- `__init__`
- `__call__`

Then create a class using this metaclass and instantiate it.

### Requirements

1. `__prepare__` must return a mapping usable as a class namespace.
2. `__new__` must show which attributes were produced by the class body.
3. `__init__` must initialize the class object normally.
4. `__call__` must log creation of ordinary instances.
5. The final assertions must prove that class creation happens before instance creation.


## Solution 1

`__prepare__` runs before the class body. `__new__` and `__init__` run after the body has populated the namespace. Later, instantiating the class causes the *metaclass* `__call__` method to run.


In [2]:
events = []

class TraceMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        events.append(("prepare", name))
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        public_names = tuple(
            key for key in namespace
            if not key.startswith("__")
        )
        events.append(("new", name, public_names))
        return super().__new__(mcls, name, bases, namespace)

    def __init__(cls, name, bases, namespace, **kwargs):
        events.append(("init", name))
        super().__init__(name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        events.append(("call", cls.__name__, args, kwargs))
        return super().__call__(*args, **kwargs)


class TracedWidget(metaclass=TraceMeta):
    category = "demo"

    def __init__(self, value):
        self.value = value

    def double(self):
        return self.value * 2


assert [event[0] for event in events] == ["prepare", "new", "init"]

widget = TracedWidget(21)

assert widget.double() == 42
assert [event[0] for event in events] == ["prepare", "new", "init", "call"]

for event in events:
    print(event)


('prepare', 'TracedWidget')
('new', 'TracedWidget', ('category', 'double'))
('init', 'TracedWidget')
('call', 'TracedWidget', (21,), {})


# Problem 2 — Inject behavior during class creation

Design a metaclass `AutoReprMeta` that injects a useful `__repr__` method into a class *only if the class did not define one itself*.

The generated representation should include the class name and selected instance attributes.

### Requirements

- A class may declare `__repr_fields__ = ("name", "age")`.
- The metaclass injects `__repr__` before creating the class.
- An explicitly defined `__repr__` must never be overwritten.
- Avoid lambdas for nontrivial injected methods; use a named nested function for readability.


## Solution 2

Mutating the namespace *before* calling `super().__new__` is usually cleaner than attaching methods afterward. This allows normal class creation machinery to see the final namespace.


In [3]:
class AutoReprMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        if "__repr__" not in namespace:
            fields = tuple(namespace.get("__repr_fields__", ()))

            def __repr__(self):
                parts = ", ".join(
                    f"{field}={getattr(self, field)!r}"
                    for field in fields
                )
                return f"{type(self).__name__}({parts})"

            namespace["__repr__"] = __repr__

        return super().__new__(mcls, name, bases, namespace)


class ReprPerson(metaclass=AutoReprMeta):
    __repr_fields__ = ("name", "age")

    def __init__(self, name, age):
        self.name = name
        self.age = age


class CustomReprPerson(metaclass=AutoReprMeta):
    def __repr__(self):
        return "<custom-repr>"


p = ReprPerson("Ada", 36)
cp = CustomReprPerson()

assert repr(p) == "ReprPerson(name='Ada', age=36)"
assert repr(cp) == "<custom-repr>"

print(repr(p))
print(repr(cp))


ReprPerson(name='Ada', age=36)
<custom-repr>


# Problem 3 — Enforce class-body naming conventions

Create a metaclass `PublicMethodNameMeta` that rejects public methods whose names are not lowercase snake_case.

Examples:

- `calculate_total` → allowed
- `_privateHelper` → ignored because it is private
- `CalculateTotal` → rejected
- `calculateTotal` → rejected

### Requirements

- Validate only callables defined directly in the new class body.
- Ignore dunder methods and private/protected names.
- Raise a clear `TypeError` during class creation.
- Do not mutate the namespace unless necessary.


## Solution 3

A metaclass is appropriate when the rule must be enforced consistently at class-definition time across many classes. For local style checks, a linter is usually simpler.


In [4]:
import re

_SNAKE_CASE = re.compile(r"^[a-z][a-z0-9_]*$")

class PublicMethodNameMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        invalid = []

        for attr_name, value in namespace.items():
            is_public = not attr_name.startswith("_")
            is_callable = callable(value)

            if is_public and is_callable and not _SNAKE_CASE.fullmatch(attr_name):
                invalid.append(attr_name)

        if invalid:
            joined = ", ".join(sorted(invalid))
            raise TypeError(
                f"{name} contains non-snake-case public method(s): {joined}"
            )

        return super().__new__(mcls, name, bases, namespace)


class GoodService(metaclass=PublicMethodNameMeta):
    def calculate_total(self):
        return 100

    def _internalHelper(self):
        return "ignored by this rule"


try:
    class BadService(metaclass=PublicMethodNameMeta):
        def calculateTotal(self):
            return 100
except TypeError as exc:
    message = str(exc)
    print(message)
    assert "calculateTotal" in message
else:
    raise AssertionError("BadService should have been rejected")

assert GoodService().calculate_total() == 100


BadService contains non-snake-case public method(s): calculateTotal


# Problem 4 — Use `__prepare__` to reject duplicate names in a class body

Normally, a later assignment silently overwrites an earlier assignment in a class body:

```python
class Example:
    value = 1
    value = 2
```

Build a metaclass `NoDuplicateNamesMeta` whose custom namespace raises an error when the same key is assigned twice.

### Important subtlety

Some valid Python patterns intentionally rebind the same name, for example:

```python
@property
def x(...): ...

@x.setter
def x(...): ...
```

So this technique is educational but often too strict for production use.


## Solution 4

`__prepare__` is the only hook that can control the mapping *while the class body is executing*. That makes it useful for preserving extra metadata or detecting writes as they happen.


In [5]:
class DuplicateNameError(TypeError):
    pass


class NoDuplicateNamespace(dict):
    def __setitem__(self, key, value):
        if key in self:
            raise DuplicateNameError(
                f"Duplicate class attribute {key!r} is not allowed"
            )
        super().__setitem__(key, value)


class NoDuplicateNamesMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return NoDuplicateNamespace()


class UniqueNamesExample(metaclass=NoDuplicateNamesMeta):
    x = 10
    y = 20

    def total(self):
        return self.x + self.y


assert UniqueNamesExample().total() == 30

try:
    class DuplicateNamesExample(metaclass=NoDuplicateNamesMeta):
        x = 10
        x = 99
except DuplicateNameError as exc:
    print(exc)
    assert "'x'" in str(exc)
else:
    raise AssertionError("Duplicate assignment should have failed")


Duplicate class attribute 'x' is not allowed


# Problem 5 — Build an inheritance-aware plugin registry

Create a registry metaclass that automatically records concrete subclasses by a unique key.

### Requirements

- Use `abc.ABCMeta` so abstract classes are supported.
- A class declares `plugin_name = "json"` (for example).
- Abstract classes are not registered.
- Duplicate keys are rejected.
- The registry should be accessible through a base class helper.
- Use `inspect.isabstract()` instead of trying to infer abstractness manually.


## Solution 5

A registry is one of the more defensible metaclass use cases because class creation is exactly the event we want to observe.


In [6]:
import inspect
from abc import ABC, ABCMeta, abstractmethod


class PluginMeta(ABCMeta):
    _registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)

        plugin_name = namespace.get("plugin_name")

        if plugin_name is not None and not inspect.isabstract(cls):
            if plugin_name in mcls._registry:
                existing = mcls._registry[plugin_name]
                raise TypeError(
                    f"Duplicate plugin_name {plugin_name!r}: "
                    f"{existing.__name__} and {name}"
                )
            mcls._registry[plugin_name] = cls

        return cls


class Serializer(ABC, metaclass=PluginMeta):
    @abstractmethod
    def dumps(self, obj):
        raise NotImplementedError

    @classmethod
    def by_name(cls, name):
        return type(cls)._registry[name]


class JsonSerializer(Serializer):
    plugin_name = "json"

    def dumps(self, obj):
        import json
        return json.dumps(obj, sort_keys=True)


class ReprSerializer(Serializer):
    plugin_name = "repr"

    def dumps(self, obj):
        return repr(obj)


assert Serializer.by_name("json") is JsonSerializer
assert Serializer.by_name("repr") is ReprSerializer
assert JsonSerializer().dumps({"b": 2, "a": 1}) == '{"a": 1, "b": 2}'

print("Registered:", sorted(PluginMeta._registry))


Registered: ['json', 'repr']


# Problem 6 — Consume custom keywords from the `class` statement

Python allows extra keywords in a class header:

```python
class User(Model, table="users"):
    ...
```

Design a metaclass that consumes a required `table=` keyword and stores it as `__table__`.

### Requirements

- The keyword must not be accidentally forwarded into `type.__new__`, which does not understand arbitrary keywords.
- The base class itself may omit the keyword.
- Concrete subclasses must provide a non-empty string.
- Show both successful and failing class declarations.


## Solution 6

Custom class keywords arrive at metaclass hooks. When you consume a keyword, remove or handle it before delegating unless the superclass explicitly accepts it.


In [7]:
class TableMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        table = kwargs.pop("table", None)

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        is_root = not bases
        if not is_root:
            if not isinstance(table, str) or not table.strip():
                raise TypeError(
                    f"{name} must declare a non-empty table=... class keyword"
                )
            cls.__table__ = table

        return cls


class TableModel(metaclass=TableMeta):
    pass


class UserRow(TableModel, table="users"):
    pass


class OrderRow(TableModel, table="orders"):
    pass


assert UserRow.__table__ == "users"
assert OrderRow.__table__ == "orders"

try:
    class BrokenRow(TableModel):
        pass
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("BrokenRow should require table=...")


BrokenRow must declare a non-empty table=... class keyword


# Problem 7 — Inject descriptors correctly and understand `__set_name__`

Build a metaclass that converts class-level declarations like:

```python
class Account(metaclass=FieldsMeta):
    balance = PositiveNumber()
```

into working validating descriptors.

Then demonstrate a subtle rule:

> If a descriptor is present in the namespace when `type.__new__` creates the class, Python automatically calls its `__set_name__`. If you attach the descriptor *after* class creation, `__set_name__` is not automatically called for you.

### Requirements

- `PositiveNumber` stores values in a private per-attribute slot.
- It rejects booleans and non-numeric values.
- It rejects values `<= 0`.
- The metaclass injects one additional descriptor before `super().__new__`.


## Solution 7

Descriptors and metaclasses are often used together. Injecting descriptors before class creation is especially important because `type.__new__` coordinates `__set_name__`.


In [8]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.public_name} must be a real number")
        if value <= 0:
            raise ValueError(f"{self.public_name} must be > 0")
        setattr(instance, self.storage_name, value)


class FieldsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace.setdefault("revision", PositiveNumber())
        return super().__new__(mcls, name, bases, namespace)


class BankAccount(metaclass=FieldsMeta):
    balance = PositiveNumber()

    def __init__(self, balance, revision=1):
        self.balance = balance
        self.revision = revision


account = BankAccount(250.0)

assert account.balance == 250.0
assert account.revision == 1
assert BankAccount.balance.public_name == "balance"
assert BankAccount.revision.public_name == "revision"

for bad in [0, -1, "100", True]:
    try:
        account.balance = bad
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"{bad!r} should have been rejected")


late = PositiveNumber()
BankAccount.credit_limit = late

assert not hasattr(late, "public_name")

late.__set_name__(BankAccount, "credit_limit")
account.credit_limit = 500
assert account.credit_limit == 500

print(account.balance, account.revision, account.credit_limit)


250.0 1 500


# Problem 8 — Collect declarative fields across inheritance

Implement an ORM-like field collector.

### Requirements

- `Field()` objects placed in a class body are declarative fields.
- Each field learns its attribute name through `__set_name__`.
- The metaclass creates `__fields__`, preserving declaration order.
- Fields inherited from base classes must appear first.
- A subclass may override an inherited field.
- Non-field attributes must not be collected.


## Solution 8

Modern Python dictionaries preserve insertion order, so a plain class namespace is normally sufficient. A custom `__prepare__` is not needed merely to preserve declaration order on supported Python versions.


In [9]:
class Field:
    _next_order = 0

    def __init__(self, *, required=True, default=None):
        self.required = required
        self.default = default
        self._order = Field._next_order
        Field._next_order += 1

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if hasattr(instance, self.storage_name):
            return getattr(instance, self.storage_name)
        if self.required and self.default is None:
            raise AttributeError(f"{self.name} has not been set")
        return self.default

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)


class FieldCollectorMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        inherited = {}

        for base in bases:
            inherited.update(getattr(base, "__fields__", {}))

        local_fields = {
            attr_name: value
            for attr_name, value in namespace.items()
            if isinstance(value, Field)
        }

        for field_name in local_fields:
            inherited.pop(field_name, None)

        combined = dict(inherited)
        combined.update(local_fields)

        namespace["__fields__"] = combined
        return super().__new__(mcls, name, bases, namespace)


class Entity(metaclass=FieldCollectorMeta):
    id = Field()


class Customer(Entity):
    name = Field()
    email = Field(required=False, default=None)
    helper_constant = 42


class PremiumCustomer(Customer):
    email = Field(required=True)
    tier = Field(default="gold", required=False)


assert list(Entity.__fields__) == ["id"]
assert list(Customer.__fields__) == ["id", "name", "email"]
assert list(PremiumCustomer.__fields__) == ["id", "name", "email", "tier"]
assert "helper_constant" not in Customer.__fields__

customer = Customer()
customer.id = 1
customer.name = "Grace"

assert customer.id == 1
assert customer.name == "Grace"
assert customer.email is None

print("Customer fields:", list(Customer.__fields__))
print("PremiumCustomer fields:", list(PremiumCustomer.__fields__))


Customer fields: ['id', 'name', 'email']
PremiumCustomer fields: ['id', 'name', 'email', 'tier']


# Problem 9 — Enforce method signatures at class-definition time

Suppose framework subclasses must implement:

```python
def handle(self, request, *, timeout=None):
    ...
```

Create a metaclass that validates the exact *shape* of `handle`.

### Requirements

- The base framework class may define the specification.
- Concrete subclasses must define `handle` directly.
- Validate positional/keyword-only structure with `inspect.signature`.
- Produce a useful error message.
- Do not call the method; inspect it.


## Solution 9

Signature validation can make framework errors fail early, but be careful not to over-constrain implementations. In many APIs, structural typing (`Protocol`) is a better fit.


In [10]:
import inspect


def signature_shape(func):
    sig = inspect.signature(func)
    return tuple(
        (param.name, param.kind, param.default is inspect._empty)
        for param in sig.parameters.values()
    )


class HandlerMeta(type):
    required_shape = (
        ("self", inspect.Parameter.POSITIONAL_OR_KEYWORD, True),
        ("request", inspect.Parameter.POSITIONAL_OR_KEYWORD, True),
        ("timeout", inspect.Parameter.KEYWORD_ONLY, False),
    )

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)

        is_framework_root = not any(isinstance(base, HandlerMeta) for base in bases)
        if not is_framework_root:
            method = namespace.get("handle")
            if method is None:
                raise TypeError(f"{name} must define handle directly")

            actual = signature_shape(method)
            if actual != mcls.required_shape:
                raise TypeError(
                    f"{name}.handle has invalid signature {inspect.signature(method)}; "
                    "expected (self, request, *, timeout=None)"
                )

        return cls


class HandlerBase(metaclass=HandlerMeta):
    pass


class GoodHandler(HandlerBase):
    def handle(self, request, *, timeout=None):
        return request, timeout


assert GoodHandler().handle("ping", timeout=2) == ("ping", 2)

try:
    class BadHandler(HandlerBase):
        def handle(self, request, timeout=None):
            return request, timeout
except TypeError as exc:
    print(exc)
    assert "expected (self, request, *, timeout=None)" in str(exc)
else:
    raise AssertionError("BadHandler should have failed signature validation")


BadHandler.handle has invalid signature (self, request, timeout=None); expected (self, request, *, timeout=None)


# Problem 10 — Resolve a metaclass conflict

Python requires the selected metaclass to be a (non-strict) subclass of the metaclasses of all base classes.

Create two independent frameworks:

- one with `AuditMeta`,
- one with `ABCMeta`.

Then demonstrate the conflict and solve it with a combined metaclass.


## Solution 10

A metaclass conflict is not resolved by base-class order. The usual solution is a compatible metaclass inheriting from all required metaclasses, assuming their cooperative `super()` implementations compose correctly.


In [11]:
from abc import ABCMeta, abstractmethod


class AuditMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__audited__ = True
        return cls


class AuditedBase(metaclass=AuditMeta):
    pass


class AbstractBase(metaclass=ABCMeta):
    @abstractmethod
    def run(self):
        raise NotImplementedError


try:
    class Conflicting(AuditedBase, AbstractBase):
        def run(self):
            return "ok"
except TypeError as exc:
    conflict_message = str(exc)
    print("Expected conflict:", conflict_message)
else:
    raise AssertionError("A metaclass conflict was expected")


class AuditABCMeta(AuditMeta, ABCMeta):
    pass


class CompatibleBase(AuditedBase, AbstractBase, metaclass=AuditABCMeta):
    def run(self):
        return "compatible"


obj = CompatibleBase()

assert obj.run() == "compatible"
assert CompatibleBase.__audited__ is True
assert isinstance(CompatibleBase, AuditMeta)
assert isinstance(CompatibleBase, ABCMeta)


Expected conflict: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


# Problem 11 — Cache instances in metaclass `__call__`

Implement a *multiton* metaclass: each unique constructor key returns the same live instance.

Example:

```python
UserSession("alice") is UserSession("alice")
```

but:

```python
UserSession("alice") is not UserSession("bob")
```

### Requirements

- Cache by positional and keyword arguments.
- Make keyword order irrelevant.
- Use `weakref.WeakValueDictionary` so the cache does not unnecessarily keep instances alive.
- Raise a clear error if arguments are unhashable.
- Explain why this should be used sparingly.


## Solution 11

Overriding metaclass `__call__` changes *instance creation* for every class using that metaclass. This is powerful but can surprise readers and complicate testing. A factory function is often more explicit.


In [12]:
import weakref


class MultitonMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__instance_cache__ = weakref.WeakValueDictionary()
        return cls

    def __call__(cls, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))

        try:
            hash(key)
        except TypeError as exc:
            raise TypeError(
                "Multiton constructor arguments must be hashable"
            ) from exc

        cached = cls.__instance_cache__.get(key)
        if cached is not None:
            return cached

        instance = super().__call__(*args, **kwargs)
        cls.__instance_cache__[key] = instance
        return instance


class UserSession(metaclass=MultitonMeta):
    def __init__(self, username, *, region="eu"):
        self.username = username
        self.region = region


a1 = UserSession("alice", region="eu")
a2 = UserSession("alice", region="eu")
a3 = UserSession("alice", **{"region": "eu"})
b = UserSession("bob", region="eu")

assert a1 is a2 is a3
assert a1 is not b

try:
    UserSession(["unhashable-list"])
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Unhashable constructor arguments should fail")

print(id(a1), id(a2), id(b))


Multiton constructor arguments must be hashable
2802879957040 2802879957040 2803148786576


# Problem 12 — Auto-wrap methods while preserving metadata

Create a metaclass `TimingMeta` that wraps every public instance method declared in a class and records call counts.

### Requirements

- Ignore dunder and private names.
- Do not wrap `staticmethod` or `classmethod` in this exercise.
- Preserve `__name__`, `__doc__`, and other metadata with `functools.wraps`.
- Avoid the classic late-binding closure bug.
- Store counts on the class in `__call_counts__`.


## Solution 12

If a metaclass transforms functions, always think about descriptor types (`staticmethod`, `classmethod`, `property`) and metadata preservation. Blindly wrapping every callable is a common bug.


In [13]:
from functools import wraps


class TimingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        call_counts = {}

        def make_wrapper(method_name, func):
            @wraps(func)
            def wrapper(self, *args, **kwargs):
                type(self).__call_counts__[method_name] += 1
                return func(self, *args, **kwargs)
            return wrapper

        transformed = dict(namespace)

        for attr_name, value in namespace.items():
            is_public = not attr_name.startswith("_")
            is_plain_function = inspect.isfunction(value)

            if is_public and is_plain_function:
                call_counts[attr_name] = 0
                transformed[attr_name] = make_wrapper(attr_name, value)

        transformed["__call_counts__"] = call_counts
        return super().__new__(mcls, name, bases, transformed)


class Calculator(metaclass=TimingMeta):
    def add(self, a, b):
        'Return a + b.'
        return a + b

    def multiply(self, a, b):
        return a * b

    def _helper(self):
        return "not wrapped"


calc = Calculator()

assert calc.add(2, 3) == 5
assert calc.add(10, 20) == 30
assert calc.multiply(4, 5) == 20

assert Calculator.__call_counts__ == {
    "add": 2,
    "multiply": 1,
}
assert Calculator.add.__name__ == "add"
assert Calculator.add.__doc__ == "Return a + b."

print(Calculator.__call_counts__)


{'add': 2, 'multiply': 1}


# Problem 13 — Prefer `__init_subclass__` when a metaclass is unnecessary

Reimplement a simple subclass registry *without* a custom metaclass.

### Requirements

- Every concrete subclass of `Command` with a `command_name` registers itself.
- Duplicate names are rejected.
- The solution must use `__init_subclass__`.
- Compare the complexity with the metaclass registry from Problem 5.


## Solution 13

`__init_subclass__` is often the best replacement for a custom metaclass when you only need to react to subclass creation and do not need to control namespace creation or the metaclass call protocol.


In [14]:
class Command:
    registry = {}

    def __init_subclass__(cls, *, command_name=None, **kwargs):
        super().__init_subclass__(**kwargs)

        if command_name is None:
            return

        if command_name in cls.registry:
            raise TypeError(f"Duplicate command_name: {command_name!r}")

        cls.command_name = command_name
        cls.registry[command_name] = cls

    def execute(self):
        raise NotImplementedError


class StartCommand(Command, command_name="start"):
    def execute(self):
        return "started"


class StopCommand(Command, command_name="stop"):
    def execute(self):
        return "stopped"


assert Command.registry["start"] is StartCommand
assert Command.registry["stop"] is StopCommand
assert Command.registry["start"]().execute() == "started"

print(Command.registry)


{'start': <class '__main__.StartCommand'>, 'stop': <class '__main__.StopCommand'>}


# Problem 14 — Prefer a class decorator for a one-off transformation

Implement the same simple feature in two ways:

1. with a metaclass,
2. with a class decorator.

The feature adds `to_dict(self)` returning non-private instance attributes.

Then compare the approaches.


## Solution 14

A class decorator is explicit at the class declaration site and avoids metaclass compatibility issues. For one-off transformations, it is often the clearer choice.


In [15]:
class ToDictMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        if "to_dict" not in namespace:
            def to_dict(self):
                return {
                    key: value
                    for key, value in vars(self).items()
                    if not key.startswith("_")
                }
            namespace["to_dict"] = to_dict

        return super().__new__(mcls, name, bases, namespace)


class MetaRecord(metaclass=ToDictMeta):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self._internal = "hidden"


def add_to_dict(cls):
    if "to_dict" not in cls.__dict__:
        def to_dict(self):
            return {
                key: value
                for key, value in vars(self).items()
                if not key.startswith("_")
            }
        cls.to_dict = to_dict
    return cls


@add_to_dict
class DecoratedRecord:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self._internal = "hidden"


expected = {"x": 1, "y": 2}

assert MetaRecord(1, 2).to_dict() == expected
assert DecoratedRecord(1, 2).to_dict() == expected

print("Metaclass:", MetaRecord(1, 2).to_dict())
print("Decorator:", DecoratedRecord(1, 2).to_dict())


Metaclass: {'x': 1, 'y': 2}
Decorator: {'x': 1, 'y': 2}


# Problem 15 — Make inherited metaclass configuration explicit

Build a metaclass that reads a `Meta` inner class:

```python
class Document(metaclass=ConfigMeta):
    class Meta:
        cache = True
        retries = 3
```

Subclasses inherit configuration and may override selected options.

### Requirements

- Supported keys: `cache`, `retries`.
- Unknown keys raise `TypeError`.
- Inherited configuration must be copied, not mutated in place.
- The inner `Meta` class should not remain as a normal runtime attribute unless you intentionally keep it.


## Solution 15

Frameworks sometimes use inner configuration classes, but explicit class keywords are often simpler for small APIs. This exercise focuses on careful inheritance and avoiding shared mutable configuration.


In [16]:
class ConfigMeta(type):
    allowed = {"cache", "retries"}
    defaults = {"cache": False, "retries": 0}

    def __new__(mcls, name, bases, namespace, **kwargs):
        inherited = dict(mcls.defaults)

        for base in bases:
            if hasattr(base, "__config__"):
                inherited.update(base.__config__)

        local_meta = namespace.pop("Meta", None)

        if local_meta is not None:
            local_options = {
                key: value
                for key, value in vars(local_meta).items()
                if not key.startswith("_")
            }

            unknown = set(local_options) - mcls.allowed
            if unknown:
                raise TypeError(
                    f"{name}.Meta contains unsupported option(s): "
                    f"{', '.join(sorted(unknown))}"
                )

            inherited.update(local_options)

        namespace["__config__"] = inherited
        return super().__new__(mcls, name, bases, namespace)


class Document(metaclass=ConfigMeta):
    class Meta:
        cache = True
        retries = 3


class FastDocument(Document):
    class Meta:
        retries = 1


class PlainDocument(Document):
    pass


assert Document.__config__ == {"cache": True, "retries": 3}
assert FastDocument.__config__ == {"cache": True, "retries": 1}
assert PlainDocument.__config__ == {"cache": True, "retries": 3}

assert Document.__config__ is not FastDocument.__config__
assert Document.__config__ is not PlainDocument.__config__
assert not hasattr(Document, "Meta")

print(Document.__config__)
print(FastDocument.__config__)


{'cache': True, 'retries': 3}
{'cache': True, 'retries': 1}


# Problem 16 — Capstone: build a mini validated model system

Build a small declarative model framework combining several metaclass techniques.

A model declaration should look like this:

```python
class Product(Model):
    sku = StringField(min_length=1)
    price = NumberField(min_value=0)
```

### Requirements

#### Fields
- implement the descriptor protocol,
- know their attribute names,
- validate assigned values,
- support defaults,
- distinguish "missing" from `None`.

#### Metaclass
- collect inherited and local fields,
- allow field overrides,
- create an immutable-ish field mapping via `MappingProxyType`,
- reject concrete models with zero fields,
- register concrete model classes by class name,
- generate a keyword-only constructor,
- reject unknown constructor arguments,
- enforce required fields.

#### Model API
- `to_dict()`
- `from_dict()`
- informative `repr`
- model lookup through the registry

### Best-practice constraints
- Keep validation inside descriptors rather than putting every concern into the metaclass.
- Use cooperative `super()`.
- Keep the generated API small and predictable.
- Do not silently swallow unknown input.


## Solution 16

This is deliberately larger than the earlier exercises. Notice the separation of responsibilities:

- descriptors validate field values,
- the metaclass assembles class-level metadata and generates the constructor,
- the base model provides reusable instance behavior.


In [17]:
from types import MappingProxyType


MISSING = object()


class ModelField:
    def __init__(self, *, default=MISSING, required=None):
        self.default = default
        self.required = (default is MISSING) if required is None else required

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def validate(self, value):
        return value

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        if hasattr(instance, self.storage_name):
            return getattr(instance, self.storage_name)

        if self.default is not MISSING:
            return self.default

        raise AttributeError(f"{self.name} is not set")

    def __set__(self, instance, value):
        validated = self.validate(value)
        setattr(instance, self.storage_name, validated)


class StringField(ModelField):
    def __init__(self, *, min_length=0, max_length=None, **kwargs):
        super().__init__(**kwargs)
        self.min_length = min_length
        self.max_length = max_length

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be str")

        if len(value) < self.min_length:
            raise ValueError(
                f"{self.name} must contain at least {self.min_length} characters"
            )

        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(
                f"{self.name} must contain at most {self.max_length} characters"
            )

        return value


class NumberField(ModelField):
    def __init__(self, *, min_value=None, max_value=None, **kwargs):
        super().__init__(**kwargs)
        self.min_value = min_value
        self.max_value = max_value

    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be int or float")

        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} must be >= {self.min_value}")

        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} must be <= {self.max_value}")

        return value


class ModelMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace, *, abstract=False, **kwargs):
        inherited_fields = {}

        for base in bases:
            inherited_fields.update(getattr(base, "__fields__", {}))

        local_fields = {
            attr_name: value
            for attr_name, value in namespace.items()
            if isinstance(value, ModelField)
        }

        for field_name in local_fields:
            inherited_fields.pop(field_name, None)

        all_fields = dict(inherited_fields)
        all_fields.update(local_fields)

        namespace["__fields__"] = MappingProxyType(all_fields)
        namespace["__abstract_model__"] = abstract

        if not abstract and bases and not all_fields:
            raise TypeError(f"Concrete model {name} must define at least one field")

        cls = super().__new__(mcls, name, bases, namespace)

        if not abstract:
            if name in mcls.registry:
                raise TypeError(f"Duplicate model class name: {name}")
            mcls.registry[name] = cls

        return cls

    def __call__(cls, **kwargs):
        unknown = set(kwargs) - set(cls.__fields__)
        if unknown:
            raise TypeError(
                f"Unknown field(s) for {cls.__name__}: "
                f"{', '.join(sorted(unknown))}"
            )

        instance = cls.__new__(cls)

        for field_name, field in cls.__fields__.items():
            if field_name in kwargs:
                setattr(instance, field_name, kwargs[field_name])
            elif field.default is not MISSING:
                setattr(instance, field_name, field.default)
            elif field.required:
                raise TypeError(
                    f"Missing required field {field_name!r} "
                    f"for {cls.__name__}"
                )

        init = cls.__init__
        if init is not object.__init__:
            init(instance)

        return instance


class Model(metaclass=ModelMeta, abstract=True):
    @classmethod
    def model_by_name(cls, name):
        return type(cls).registry[name]

    @classmethod
    def from_dict(cls, data):
        if not isinstance(data, dict):
            raise TypeError("from_dict expects a dict")
        return cls(**data)

    def to_dict(self):
        result = {}

        for field_name in self.__fields__:
            try:
                result[field_name] = getattr(self, field_name)
            except AttributeError:
                pass

        return result

    def __repr__(self):
        values = ", ".join(
            f"{name}={value!r}"
            for name, value in self.to_dict().items()
        )
        return f"{type(self).__name__}({values})"


class Product(Model):
    sku = StringField(min_length=1)
    name = StringField(min_length=1, max_length=80)
    price = NumberField(min_value=0)
    stock = NumberField(min_value=0, default=0)


class DiscountedProduct(Product):
    discount = NumberField(min_value=0, max_value=1, default=0.0)


product = Product(
    sku="SKU-001",
    name="Mechanical Keyboard",
    price=129.99,
)

assert product.stock == 0
assert product.to_dict() == {
    "sku": "SKU-001",
    "name": "Mechanical Keyboard",
    "price": 129.99,
    "stock": 0,
}
assert repr(product).startswith("Product(")

clone = Product.from_dict(product.to_dict())
assert clone.to_dict() == product.to_dict()

discounted = DiscountedProduct(
    sku="SKU-002",
    name="Mouse",
    price=49.99,
    discount=0.20,
)

assert list(DiscountedProduct.__fields__) == [
    "sku",
    "name",
    "price",
    "stock",
    "discount",
]
assert Model.model_by_name("Product") is Product
assert Model.model_by_name("DiscountedProduct") is DiscountedProduct

try:
    Product(
        sku="X",
        name="Bad Input Example",
        price=1,
        does_not_exist=123,
    )
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Unknown fields should be rejected")

try:
    Product(
        sku="X",
        name="Missing Price",
    )
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Missing required fields should be rejected")

try:
    Product(
        sku="X",
        name="Negative Price",
        price=-5,
    )
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("Negative price should be rejected")

print(product)
print(discounted)
print("Registry:", sorted(ModelMeta.registry))


Unknown field(s) for Product: does_not_exist
Missing required field 'price' for Product
price must be >= 0
Product(sku='SKU-001', name='Mechanical Keyboard', price=129.99, stock=0)
DiscountedProduct(sku='SKU-002', name='Mouse', price=49.99, stock=0, discount=0.2)
Registry: ['DiscountedProduct', 'Product']


# Problem 17 — Route collection with method decorators + metaclass assembly

Build a tiny routing framework.

A decorator marks methods with route metadata:

```python
@route("GET", "/users")
def users(self): ...
```

The metaclass must collect routes from base classes and subclasses.

### Requirements

- The decorator itself should only attach metadata to the function.
- The metaclass assembles `__routes__`.
- Subclasses inherit routes.
- A subclass route with the same `(method, path)` replaces the inherited route.
- Detect duplicate routes declared in the same class body.
- Keep routing metadata separate from method execution.


## Solution 17

This is a useful pattern: decorators mark individual methods, while the metaclass assembles class-wide metadata once the whole class body is available.


In [18]:
def route(http_method, path):
    http_method = http_method.upper()

    def decorator(func):
        func.__route__ = (http_method, path)
        return func

    return decorator


class RouteMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        routes = {}

        for base in bases:
            routes.update(getattr(base, "__routes__", {}))

        local_seen = set()

        for attr_name, value in namespace.items():
            route_key = getattr(value, "__route__", None)
            if route_key is None:
                continue

            if route_key in local_seen:
                raise TypeError(
                    f"{name} declares duplicate route {route_key}"
                )

            local_seen.add(route_key)
            routes[route_key] = attr_name

        namespace["__routes__"] = MappingProxyType(routes)
        return super().__new__(mcls, name, bases, namespace)


class BaseController(metaclass=RouteMeta):
    @route("GET", "/health")
    def health(self):
        return {"ok": True}


class UserController(BaseController):
    @route("GET", "/users")
    def list_users(self):
        return ["Ada", "Grace"]

    @route("POST", "/users")
    def create_user(self):
        return {"created": True}


class CustomHealthController(UserController):
    @route("GET", "/health")
    def health_v2(self):
        return {"ok": True, "version": 2}


assert BaseController.__routes__[("GET", "/health")] == "health"
assert UserController.__routes__[("GET", "/users")] == "list_users"
assert UserController.__routes__[("POST", "/users")] == "create_user"
assert CustomHealthController.__routes__[("GET", "/health")] == "health_v2"

controller = CustomHealthController()
handler_name = controller.__routes__[("GET", "/health")]
response = getattr(controller, handler_name)()

assert response["version"] == 2

print(dict(CustomHealthController.__routes__))


{('GET', '/health'): 'health_v2', ('GET', '/users'): 'list_users', ('POST', '/users'): 'create_user'}


# Problem 18 — A "final class" rule, then refactor it to the simpler mechanism

First, enforce a `__final__ = True` convention with a metaclass: no subclass may inherit from a final base.

Then implement the same policy with `__init_subclass__` and compare the two.

This problem is intentionally about *choosing the simpler mechanism* after learning the advanced one.


## Solution 18A — Metaclass version


In [19]:
class FinalMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        final_bases = [
            base.__name__
            for base in bases
            if getattr(base, "__final__", False)
        ]

        if final_bases:
            raise TypeError(
                f"{name} cannot inherit from final class(es): "
                f"{', '.join(final_bases)}"
            )

        return super().__new__(mcls, name, bases, namespace)


class FinalBase(metaclass=FinalMeta):
    __final__ = True


try:
    class IllegalChild(FinalBase):
        pass
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Subclassing FinalBase should fail")


IllegalChild cannot inherit from final class(es): FinalBase


## Solution 18B — `__init_subclass__` version

The metaclass works, but the subclass hook is smaller and avoids metaclass conflicts.


In [20]:
class FinalByHook:
    __final__ = False

    def __init_subclass__(cls, **kwargs):
        for base in cls.__bases__:
            if getattr(base, "__final__", False):
                raise TypeError(
                    f"{cls.__name__} cannot inherit from final class {base.__name__}"
                )

        super().__init_subclass__(**kwargs)


class HookFinalClass(FinalByHook):
    __final__ = True


try:
    class HookIllegalChild(HookFinalClass):
        pass
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Subclassing HookFinalClass should fail")


HookIllegalChild cannot inherit from final class HookFinalClass


# Extra challenge set — Try these before reading the hints

These are intentionally left as extension exercises rather than full duplicate implementations.

### Challenge A — Metaclass-based test discovery
Collect methods whose names start with `test_` into `__tests__`, preserving inheritance and allowing overrides.

### Challenge B — Enum-like constants
Build a metaclass that converts uppercase class attributes into immutable member objects. Then compare your design with the standard-library `enum.Enum`.

### Challenge C — SQL schema generator
Reuse the capstone `ModelField` hierarchy and add field metadata such as SQL type, nullable, and primary-key flags. Generate a `CREATE TABLE` statement at class creation.

### Challenge D — API version validation
Require subclasses to declare semantic versions such as `"2.4.1"` and prevent a subclass from declaring a version lower than its base class.

### Challenge E — Cooperative metaclasses
Create two metaclasses that both modify class creation, then combine them. Make each implementation use `super()` correctly so both behaviors survive multiple inheritance.

### Challenge F — Metaclass + `__slots__`
Generate `__slots__` from declared fields. Carefully handle inheritance and explain why blindly adding slots can break classes.

### Challenge G — Generic aliases
Investigate how `__class_getitem__` differs from metaclass `__getitem__`, and implement tiny examples of both.

### Challenge H — Dynamic class factories
Implement a normal factory function using `type(name, bases, namespace)`, then reimplement the same feature declaratively through a metaclass. Compare readability.


# Best-practice checklist for production metaclasses

Use this checklist before shipping a metaclass:

1. **Ask whether a simpler feature works first.** Consider a function, class decorator, descriptor, `__init_subclass__`, `dataclasses`, `ABC`, or `Protocol`.
2. **Keep the metaclass focused on class-level concerns.** Put instance validation in descriptors or normal methods.
3. **Use cooperative `super()`.** This makes metaclasses more likely to compose with other frameworks.
4. **Accept and handle `**kwargs` carefully.** Class header keywords flow into metaclass hooks.
5. **Do not silently overwrite user-defined attributes.** Prefer explicit conflict handling.
6. **Preserve metadata when wrapping methods.** Use `functools.wraps`.
7. **Know descriptor timing.** Descriptors injected before `type.__new__` get automatic `__set_name__`.
8. **Avoid unnecessary `__prepare__`.** Normal dictionaries already preserve insertion order.
9. **Fail early with clear messages.** If the metaclass enforces a rule, errors should identify the class and offending member.
10. **Test inheritance.** Most subtle metaclass bugs appear when subclasses override or combine behavior.
11. **Test multiple inheritance.** Especially when custom metaclasses may meet `ABCMeta` or framework metaclasses.
12. **Keep magic discoverable.** Document generated methods, registries, and class keywords.
13. **Avoid global mutable state where possible.** If a registry is global, define its ownership and reset strategy for tests.
14. **Be cautious with metaclass `__call__`.** It changes normal object construction and can make factories, caching, or singleton behavior less obvious.
15. **Prefer standard-library solutions when available.** `Enum`, `dataclass`, `ABC`, and `Protocol` already solve many problems safely.


# Compact review: which hook should I use?

| Need | Usually use |
|---|---|
| Customize namespace before class body executes | metaclass `__prepare__` |
| Transform/validate class namespace before class exists | metaclass `__new__` |
| Initialize metadata after class object exists | metaclass `__init__` |
| Control creation of instances of classes using the metaclass | metaclass `__call__` |
| React to subclass creation only | `__init_subclass__` |
| Transform one class explicitly | class decorator |
| Validate/manage individual attributes | descriptor |
| Enforce abstract methods | `ABC` / `ABCMeta` |
| Structural interface checking | `typing.Protocol` |

A good rule: if you do not specifically need control over *class creation*, you probably do not need a metaclass.


In [21]:
# Final notebook smoke test

checks = {
    "TraceMeta": TraceMeta,
    "AutoReprMeta": AutoReprMeta,
    "PluginMeta": PluginMeta,
    "FieldCollectorMeta": FieldCollectorMeta,
    "HandlerMeta": HandlerMeta,
    "AuditABCMeta": AuditABCMeta,
    "MultitonMeta": MultitonMeta,
    "TimingMeta": TimingMeta,
    "ConfigMeta": ConfigMeta,
    "ModelMeta": ModelMeta,
    "RouteMeta": RouteMeta,
    "FinalMeta": FinalMeta,
}

assert all(isinstance(meta, type) for meta in checks.values())
assert Product.__fields__["price"].min_value == 0
assert ("GET", "/users") in UserController.__routes__

print(f"All {len(checks)} metaclass examples are available.")
print("Notebook smoke test passed.")


All 12 metaclass examples are available.
Notebook smoke test passed.
